# One-Day VNP46A2 / VNP46A1 / VJ146A2 Stray-Light Comparison - Threshold-1 Scale

This notebook compares the A1/VJ diagnostic outputs for the same review dates used in `stray_light_blackmarbler_redownload_comparison.ipynb`: `2023-10-20`, `2023-02-28`, and `2023-01-29`.

All map panels use a fixed radiance color scale capped at `1.0`, matching the production pipeline lit threshold (`rad > 1`). Pixels above the threshold saturate at the maximum color, so the maps show what enters the lit/not-lit decision.

It does not download data and does not change production outputs.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATES = ["2023-10-20", "2023-02-28", "2023-01-29"]

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "STRAY_LIGHT_FIX_PLAN.md").exists() and (path / "blackmarbler").exists():
            return path
        nested = path / "6-codebases" / "repos" / "Reliability-Assessment"
        if (nested / "STRAY_LIGHT_FIX_PLAN.md").exists() and (nested / "blackmarbler").exists():
            return nested
    raise FileNotFoundError("Could not locate Reliability-Assessment repo root.")

ROOT = find_repo_root(Path.cwd())
OUT_DIR = ROOT / "blackmarbler" / "out_vnp46a2_sa_daily" / "qa_straylight_validation" / "one_day_a1_vj"
THRESHOLD_PANEL_DIR = OUT_DIR / "png" / "threshold1_scale"
SEPARATE_PANEL_DIR = THRESHOLD_PANEL_DIR / "separate_panels"

def paths_for_date(date: str) -> dict:
    return {
        "summary": OUT_DIR / f"one_day_a1_vj_summary_{date}.csv",
        "masks": OUT_DIR / f"one_day_a1_vj_mask_shares_{date}.csv",
        "panel": THRESHOLD_PANEL_DIR / f"threshold1_comparison_{date}.png",
    }

print(f"Repo root: {ROOT}")
print(f"Diagnostic outputs: {OUT_DIR}")
print(f"Dates: {', '.join(DATES)}")

In [ ]:
required = []
for date in DATES:
    p = paths_for_date(date)
    required.extend([p["summary"], p["masks"], p["panel"]])
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run stray_light_one_day_a1_vj_diagnostic.R first. Missing:\n" + "\n".join(str(p) for p in missing))

summary = pd.concat([pd.read_csv(paths_for_date(date)["summary"]) for date in DATES], ignore_index=True)
masks = pd.concat([pd.read_csv(paths_for_date(date)["masks"]) for date in DATES], ignore_index=True)
summary["date"] = summary["date"].astype(str)
masks["date"] = masks["date"].astype(str)

summary

## Side-by-Side Map Panel

The panel uses a fixed `0–1` radiance color scale across scenarios. White areas are invalid/missing; saturated bright areas are at or above the production lit threshold.

In [ ]:
for date in DATES:
    img = plt.imread(paths_for_date(date)["panel"])
    fig, ax = plt.subplots(figsize=(16, 9.6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"One-day A1/VJ diagnostic: {date}")
    plt.show()

## Separate Readable Panels

These panels are rendered directly from the scenario GeoTIFFs with `0–1` radiance scaling, shown one scenario at a time so the maps and legends are easier to inspect. They are saved as standalone PNGs in `png/threshold1_scale/separate_panels/`.

In [ ]:
panel_files = {
    "Current VNP46A2": "current_vnp46a2",
    "VNP46A2 + A1 stray bit": "vnp46a2_a1_stray_bit",
    "VNP46A2 + A1 strict DNB bits": "vnp46a2_a1_strict_dnb_bits",
    "VJ146A2 candidate C2 mask": "vj146a2_candidate_c2_mask",
    "Blend: VNP strict else VJ": "blend_vnp_strict_else_vj",
}

for date in DATES:
    print(f"=== {date} ===")
    for scenario, stem in panel_files.items():
        out_png = SEPARATE_PANEL_DIR / f"{stem}_{date}_threshold1.png"
        if not out_png.exists():
            raise FileNotFoundError(out_png)
        image = plt.imread(out_png)

        metrics = summary.loc[(summary["date"] == date) & (summary["scenario"] == scenario), ["valid_share", "p_lit", "lit_share_all"]]
        fig, ax = plt.subplots(figsize=(12, 7.5))
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(f"{date} - {scenario}")
        plt.show()

        print(metrics.to_string(index=False))
        print(f"source: {out_png}\n")

## Quantitative Comparison

`p_lit` is lit share among valid pixels. `lit_share_all` is lit share over all pixels in the South Africa-minus-Lesotho raster.

In [ ]:
display_cols = ["date", "scenario", "valid_share", "p_lit", "lit_share_all", "mean_rad_valid", "n_valid_pixels", "n_lit_pixels"]
summary[display_cols].sort_values(["date", "scenario"])

In [ ]:
comparisons = []
for date in DATES:
    date_rows = summary.loc[summary["date"] == date].copy()
    baseline = date_rows.loc[date_rows["scenario"] == "Current VNP46A2"].iloc[0]
    date_rows["delta_valid_share_vs_current"] = date_rows["valid_share"] - baseline["valid_share"]
    date_rows["delta_p_lit_vs_current"] = date_rows["p_lit"] - baseline["p_lit"]
    date_rows["delta_lit_share_all_vs_current"] = date_rows["lit_share_all"] - baseline["lit_share_all"]
    comparisons.append(date_rows)

comparison = pd.concat(comparisons, ignore_index=True)
comparison[["date", "scenario", "delta_valid_share_vs_current", "delta_p_lit_vs_current", "delta_lit_share_all_vs_current"]]

## A1 QF_DNB Mask Check

This confirms whether the VNP46A1 DNB QA bits actually reject any pixels on this date.

In [ ]:
masks

## Takeaway

At the pipeline threshold scale, contaminated VNP46A2 scenes show large saturated areas. VJ146A2 is much cleaner on the contaminated review dates, but the clean control still argues against applying VJ replacement globally without a contamination gate.